**METRICS — Optimized joins on the 100M-row fact**

Every cell below applies the same playbook:

- **Project both sides before the join** — fact narrowed to ~7 columns; each dim narrowed to `(join_key + only the columns this metric uses)`.
- **`broadcast()` on every small dim** — forces a `BroadcastHashJoin`, no shuffle on the fact side.
- **Aggregate after enrichment** — partial aggregation runs inside each map task; only the small partial results shuffle to the final reducer.
- **`.explain("formatted")` printed for every plan** — verify `BroadcastHashJoin`, no `Exchange` on the fact side before the join.

**Shuffle reduction summary:** at 100M rows each avoided shuffle saves writing/reading ~5–10GB of intermediate data. Broadcasting the dims (each <1MB) trades a tiny per-executor memory cost for eliminating the fact-side shuffle entirely. Partial aggregation reduces the final shuffle to `group_cardinality × num_partitions` rows — typically a few thousand for region/category/date groupings.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

# Read RAW fact_sales (not pre-enriched). The point of this notebook is to demonstrate
# query-time join optimization — joining per-metric lets each metric project only the
# columns it actually needs from each dim. A pre-enriched table forces every metric to
# read every column, which wastes I/O at 100M-row scale.
fact_sales = spark.table("fact_sales")

# Read each dim once. Each metric below narrows them further before joining.
# Sizes (rows): dim_store=200, dim_product=5_000, dim_promotion=50.
# All three are tiny — well under spark.sql.autoBroadcastJoinThreshold (default 10MB).
# We never need dim_customer for any metric here, so we don't load it.
dim_store = spark.table("dim_store")
dim_product = spark.table("dim_product")
dim_promotion = spark.table("dim_promotion")

# Helper: net_revenue formula reused by every metric. Computed inline (no UDF) so Catalyst
# can fold it into the same Project node as the join output — zero extra stages.
#   net_revenue = quantity * unit_price * (1 - coalesce(discount_pct, 0))
# The coalesce handles the ~70% of rows where promotion_id is NULL → no promo row matches → discount_pct is NULL.
def net_revenue_expr():
    return F.col("quantity") * F.col("unit_price") * (F.lit(1.0) - F.coalesce(F.col("discount_pct"), F.lit(0.0)))

Revenue by region ->shows business performance by global region

In [0]:
# Revenue by region.
# Joins needed: fact -> dim_store (region), fact -> dim_promotion (discount_pct).
# We do NOT need dim_product for this metric, so it's not joined → no wasted shuffle/broadcast.

# Step 1: project the fact down to ONLY the columns we actually use.
# Without this, Spark would carry every fact column through the broadcast hash join,
# inflating each task's heap usage. At 100M rows, dropping unused columns saves ~30–40%
# of in-flight memory and lets the broadcast hash table stay tighter on each executor.
fact_proj = fact_sales.select(
    "store_id",        # join key for dim_store
    "promotion_id",    # join key for dim_promotion
    "quantity",        # net_revenue input
    "unit_price",      # net_revenue input
)

# Step 2: project each dim to (join_key + ONLY columns we need from it).
# dim_store has 4 columns; we only need region. The narrower the broadcast, the less
# memory each executor consumes for the broadcast hash table.
store_proj = dim_store.select("store_id", "region")
promo_proj = dim_promotion.select("promotion_id", "discount_pct")

# Step 3: broadcast joins. Both dims are tiny (<1MB each). broadcast() forces a
# BroadcastHashJoin, which means:
#   - dim is sent to every executor once (small one-time network cost),
#   - the 100M-row fact is NEVER shuffled — it streams through each task locally.
# Compare to a default sort-merge join, which would shuffle BOTH sides by the join key,
# i.e., ~5–10GB of fact data over the network per join. We're saving two such shuffles here.
revenue_by_region = (
    fact_proj
    .join(broadcast(store_proj), "store_id", "left")
    .join(broadcast(promo_proj), "promotion_id", "left")
    # Step 4: aggregate AFTER enrichment.
    # Spark performs partial aggregation (HashAggregate) inside each map task before the
    # final Exchange. So per-task we go from ~500K rows → ~5 rows (one per region).
    # The shuffle that feeds the final aggregation only carries num_partitions × 5 = 1000 rows.
    .groupBy("region")
    .agg(F.round(F.sum(net_revenue_expr()), 2).alias("total_net_revenue"))
    .orderBy(F.desc("total_net_revenue"))
)

# Inspect the physical plan. Look for:
#   - "BroadcastHashJoin" (not SortMergeJoin) on both join nodes
#   - "BroadcastExchange" feeding only the dim side of each join
#   - No "Exchange" on the fact side before the final HashAggregate's shuffle
#   - Two HashAggregate nodes: a partial one before the Exchange, a final one after
revenue_by_region.explain("formatted")
display(revenue_by_region)

revenue by product category

In [0]:
# Revenue by product category.
# Joins: fact -> dim_product (category), fact -> dim_promotion (discount_pct).
# dim_store is irrelevant here — skipping it removes a join entirely.

# Project fact to ONLY needed columns. Each dropped column means less data flowing through
# the join's output rows on the executor side.
fact_proj = fact_sales.select("product_id", "promotion_id", "quantity", "unit_price")

# Narrow dims: dim_product has 4 columns, we only need category. dim_promotion just needs
# discount_pct. Smaller broadcast hash tables = less per-executor memory.
product_proj = dim_product.select("product_id", "category")
promo_proj = dim_promotion.select("promotion_id", "discount_pct")

revenue_by_category = (
    fact_proj
    # broadcast() forces BroadcastHashJoin, eliminating the fact-side shuffle that a
    # SortMergeJoin would require (~5–10GB of network traffic at 100M rows).
    .join(broadcast(product_proj), "product_id", "left")
    .join(broadcast(promo_proj), "promotion_id", "left")
    # Aggregate after enrichment. Partial aggregation collapses 100M rows down to
    # 8 (one per category) PER TASK, so the shuffle to the final aggregator carries
    # num_partitions × 8 ≈ 1.6K rows total — negligible compared to the unaggregated case.
    .groupBy("category")
    .agg(F.round(F.sum(net_revenue_expr()), 2).alias("total_net_revenue"))
    .orderBy(F.desc("total_net_revenue"))
)

# Verify in the plan: BroadcastHashJoin on both joins, BroadcastExchange only on dim sides,
# no Exchange on fact side before the partial HashAggregate.
revenue_by_category.explain("formatted")
display(revenue_by_category)

Gross margin by category

In [0]:
# Gross margin by category.
# Joins: fact -> dim_product (category, unit_cost), fact -> dim_promotion (discount_pct).
# Both gross_margin and net_revenue need promo for the discount, and product for unit_cost.

fact_proj = fact_sales.select("product_id", "promotion_id", "quantity", "unit_price")

# dim_product needs TWO columns this time: category for the group, unit_cost for the math.
# Still tiny (~5K rows × 2 cols ≈ a few hundred KB) — well under broadcast threshold.
product_proj = dim_product.select("product_id", "category", "unit_cost")
promo_proj = dim_promotion.select("promotion_id", "discount_pct")

margin_by_category = (
    fact_proj
    .join(broadcast(product_proj), "product_id", "left")
    .join(broadcast(promo_proj), "promotion_id", "left")
    # Compute net_revenue and cost in a single Project (Catalyst fuses these withColumns).
    # Doing both metric inputs in one pass means we don't re-scan the row to compute them
    # separately — they share the per-row read of quantity, unit_price, unit_cost.
    .withColumn("net_revenue", net_revenue_expr())
    .withColumn("cost", F.col("quantity") * F.col("unit_cost"))
    # Aggregate after enrichment. Partial HashAggregate collapses each task to 8 rows
    # (one per category), so the shuffle is trivial. We compute three sums in the same
    # aggregation pass so we only scan the post-join rows once.
    .groupBy("category")
    .agg(
        F.round(F.sum("net_revenue"), 2).alias("total_net_revenue"),
        F.round(F.sum(F.col("net_revenue") - F.col("cost")), 2).alias("gross_margin"),
        # gross_margin_pct computed from the two sums above. Doing the division INSIDE
        # the agg(...) lets Catalyst share the partial sums of net_revenue and (net_revenue-cost)
        # — no need for a self-join or window function.
        F.round(F.sum(F.col("net_revenue") - F.col("cost")) / F.sum("net_revenue"), 4).alias("gross_margin_pct"),
    )
    .orderBy(F.desc("gross_margin"))
)

margin_by_category.explain("formatted")
display(margin_by_category)

Average basket value

In [0]:
# Average basket value.
# In our schema each transaction_id is unique → "basket_revenue per transaction" is just
# the row's net_revenue. The original cell did a groupBy("transaction_id") with sum() — at
# 100M unique keys that's a 100M-group shuffle, which is enormous and pointless.
#
# Optimization: skip the per-transaction groupBy entirely and compute the global avg
# in a single full-table aggregation. That's a single shuffle of partial sums (~200 rows
# total), regardless of fact size.

# Project to bare minimum: we only need promotion_id (for discount lookup) plus the two
# numeric inputs. No store/product/customer joins needed — none of those affect basket value.
fact_proj = fact_sales.select("promotion_id", "quantity", "unit_price")

# dim_promotion is the only join — broadcast it to avoid shuffling the 100M-row fact.
promo_proj = dim_promotion.select("promotion_id", "discount_pct")

avg_basket_value = (
    fact_proj
    .join(broadcast(promo_proj), "promotion_id", "left")
    # Single global aggregation. agg() with no groupBy() = one row out, but partial
    # aggregation still runs per task, so the shuffle to the final reducer is just
    # num_partitions × 1 = ~200 rows total.
    .agg(F.round(F.avg(net_revenue_expr()), 2).alias("avg_basket_value"))
)

# Plan should show: BroadcastHashJoin → partial HashAggregate (per task) → Exchange
# (tiny, ~200 rows of partial sums) → final HashAggregate. No fact-side shuffle anywhere.
avg_basket_value.explain("formatted")
display(avg_basket_value)

Daily revenue trend

In [0]:
# Daily revenue trend.
# Joins: fact -> dim_promotion (discount_pct).
# transaction_date already lives on the fact, so no date-dim join needed. We don't
# touch dim_store/dim_product/dim_customer — none of them affect a daily revenue total.

# Project fact to ONLY needed columns. transaction_date is the partition column on the
# Delta write, so reading just it + the three measure-input columns lets Spark prune
# everything else in the columnar Parquet read — significant I/O savings at 100M rows.
fact_proj = fact_sales.select("transaction_date", "promotion_id", "quantity", "unit_price")

# Narrow dim_promotion to (join_key, discount_pct). 50 rows × 2 cols = trivially broadcastable.
promo_proj = dim_promotion.select("promotion_id", "discount_pct")

daily_revenue = (
    fact_proj
    # broadcast() forces BroadcastHashJoin → no shuffle of the 100M-row fact.
    # A default sort-merge join here would shuffle ~5GB of fact data; we save that entirely.
    .join(broadcast(promo_proj), "promotion_id", "left")
    # Aggregate after enrichment. Partial HashAggregate collapses each task to ~365 rows
    # (one per date in CY2024), so the shuffle to the final aggregator carries
    # num_partitions × 365 ≈ 73K rows — tiny compared to 100M unaggregated rows.
    .groupBy("transaction_date")
    .agg(F.round(F.sum(net_revenue_expr()), 2).alias("daily_net_revenue"))
    .orderBy("transaction_date")
)

# Verify in plan: BroadcastHashJoin (not SortMergeJoin), BroadcastExchange only on dim side,
# no Exchange on fact side before the partial HashAggregate, then the final tiny shuffle.
daily_revenue.explain("formatted")
display(daily_revenue)


Top 10 stores by revenue

In [0]:
# Top 10 stores by revenue.
# Joins: fact -> dim_store (country, region, store_type), fact -> dim_promotion (discount_pct).
# We don't touch dim_product/dim_customer — neither affects a per-store revenue rollup.
#
# Optimization note: we group by store_id FIRST (revenue is determined entirely by store_id),
# THEN join the store attributes onto the small post-aggregation result. This avoids carrying
# country/region/store_type through the 100M-row join — they only need to ride along with
# the 200 aggregated store rows at the end.

fact_proj = fact_sales.select("store_id", "promotion_id", "quantity", "unit_price")

# dim_promotion narrowed to (join_key, discount_pct). dim_store gets joined LATER (after agg)
# so we don't even need to broadcast it through the 100M-row stage.
promo_proj = dim_promotion.select("promotion_id", "discount_pct")
store_proj = dim_store.select("store_id", "country", "region", "store_type")

# Step 1: enrich with discount, aggregate to per-store totals (200 rows out).
# This is the only stage that touches the 100M-row fact — and it only carries 5 columns.
per_store_revenue = (
    fact_proj
    .join(broadcast(promo_proj), "promotion_id", "left")
    # Partial HashAggregate collapses each task to ~200 rows (one per store_id).
    # Final shuffle: num_partitions × 200 ≈ 40K rows, vs 100M unaggregated.
    .groupBy("store_id")
    .agg(F.round(F.sum(net_revenue_expr()), 2).alias("total_net_revenue"))
)

# Step 2: join store attributes onto the 200-row result. At this scale the broadcast is
# essentially free — and crucially, country/region/store_type never traveled through the
# big-fact join. This is a pure metadata enrichment on a tiny intermediate result.
top_10_stores = (
    per_store_revenue
    .join(broadcast(store_proj), "store_id", "left")
    .orderBy(F.desc("total_net_revenue"))
    .limit(10)
)

# Plan should show: BroadcastHashJoin with promotion → partial HashAggregate (per task,
# ~200 rows) → small Exchange → final HashAggregate → BroadcastHashJoin with store
# (200×200 rows, trivial) → TakeOrderedAndProject for the top-10. No fact-side shuffle.
top_10_stores.explain("formatted")
display(top_10_stores)


Promotion performance

In [0]:
# Promotion performance.
# Joins: fact -> dim_promotion (promotion_type, discount_pct).
# Single join — store/product/customer don't affect a promotion-type rollup.
#
# Subtlety: 70% of fact rows have NULL promotion_id (most transactions aren't promoted).
# A LEFT join keeps those rows; their promotion_type/discount_pct will be NULL and we
# group them under a "no_promotion" bucket so they show up in the report rather than
# being silently dropped (which an inner join would do).

# Project to ONLY needed columns. Carrying anything else through the join would inflate
# the post-join row width unnecessarily.
fact_proj = fact_sales.select("promotion_id", "quantity", "unit_price")

# dim_promotion narrowed to the two columns this metric uses. 50 rows × 3 cols, trivial broadcast.
promo_proj = dim_promotion.select("promotion_id", "promotion_type", "discount_pct")

promotion_performance = (
    fact_proj
    # broadcast() forces BroadcastHashJoin → no shuffle of the 100M-row fact.
    # SortMergeJoin alternative would shuffle ~5GB of fact data on promotion_id.
    .join(broadcast(promo_proj), "promotion_id", "left")
    # Compute gross_revenue / discount_amount / net_revenue once in a single Project
    # so all three sums share the same per-row scan. coalesce() handles unpromoted rows
    # (NULL discount_pct → 0). Catalyst fuses these withColumn calls into one project node.
    .withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
    .withColumn(
        "discount_amount",
        F.col("gross_revenue") * F.coalesce(F.col("discount_pct"), F.lit(0.0)),
    )
    .withColumn("net_revenue", F.col("gross_revenue") - F.col("discount_amount"))
    # Bucket NULL promotion_type as "no_promotion" so unpromoted rows show up explicitly
    # rather than getting a NULL group key (which display() will render as a blank row).
    .withColumn("promotion_type", F.coalesce(F.col("promotion_type"), F.lit("no_promotion")))
    # Aggregate after enrichment. promotion_type has 6 distinct values (5 real + no_promotion),
    # so partial HashAggregate collapses each task to 6 rows. Final shuffle:
    # num_partitions × 6 ≈ 1.2K rows — negligible.
    # All four aggregations share the same scan of the post-join rows.
    .groupBy("promotion_type")
    .agg(
        F.count("*").alias("transaction_lines"),
        F.round(F.sum("gross_revenue"), 2).alias("gross_revenue"),
        F.round(F.sum("discount_amount"), 2).alias("total_discount"),
        F.round(F.sum("net_revenue"), 2).alias("net_revenue"),
    )
    .orderBy(F.desc("net_revenue"))
)

# Plan should show: BroadcastHashJoin → partial HashAggregate → small Exchange → final
# HashAggregate. No SortMergeJoin, no fact-side shuffle before the partial aggregation.
promotion_performance.explain("formatted")
display(promotion_performance)


## Skew handling — tiered response by severity

The fact generator deliberately sends ~50% of all transactions to `store_id = 1`. But before reaching for a fix, decide *which* fix. Skew tolerance varies wildly by operation:

| Operation | Skew impact | Why |
|---|---|---|
| `groupBy(skewed_key).sum/count/avg` | **small** | Partial HashAggregate collapses each task to `num_keys` rows before the shuffle. Final-stage straggler exists but carries little data. |
| `groupBy(skewed_key)` with `countDistinct` / `percentile_approx` / `collect_list` | **large** | Partial state per group is big; doesn't compress before shuffle. |
| `Window` partitioned by skewed key | **large** | Each partition processed by one task; the hot partition becomes the straggler. |
| `Join` on skewed key (non-broadcast) | **very large** | Both sides shuffled to the skewed reducer. Classic OOM/straggler case. |

### Decision framework

1. **Detect** — `groupBy(key).count()`, then derive `max/median ratio` and `hot_key_share`. (Cell 17.)
2. **Classify** — combine severity with operation type:

   | Severity | Op type | Recommended tier |
   |---|---|---|
   | `max/median < 5x` | any | **None** — let partial aggregation handle it. |
   | `5x – 50x` | join (non-broadcast) | **Tier 1 — AQE skew-join.** Just enable two configs. |
   | `5x – 50x` | wide aggregation / window | **Tier 2 — Salting.** AQE doesn't handle groupBy skew. |
   | `> 50x` | any | **Tier 2 — Salting.** AQE's split factor isn't aggressive enough. |
   | `hot_key_share > 80%` | any | **Tier 3 — Split hot-key processing.** Salting still leaves cold partitions starving; process the hot key as its own query path. |

3. **Apply** — escalate only as far as needed. Each tier costs more code/complexity than the previous one; don't pay for what you don't need.

The remaining cells implement each tier with its own demo.

In [ ]:
# Skew detection + tier classification.
# Output of this cell DRIVES which of the next three cells you should run.

# --- Step 1: per-key row counts (the diagnostic groupBy-count) -----------------------
# count(*) partials are 8 bytes each, so even though we're grouping on the skewed key,
# the final shuffle carries num_partitions × num_keys ≈ 40K rows — small enough that the
# diagnostic itself isn't bottlenecked by the skew it's measuring.
store_counts = (
    fact_sales
    .groupBy("store_id")
    .agg(F.count("*").alias("row_count"))
)

# --- Step 2: severity metrics --------------------------------------------------------
total_rows = fact_sales.count()  # cheap on Delta — uses file-level row-count metadata

skew_summary = (
    store_counts
    .agg(
        F.max("row_count").alias("max_rows"),
        F.expr("percentile_approx(row_count, 0.5)").alias("median_rows"),
        F.min("row_count").alias("min_rows"),
        F.count("*").alias("num_distinct_stores"),
    )
    .withColumn("hot_key_share_pct", F.round(F.col("max_rows") / F.lit(total_rows) * F.lit(100), 2))
    .withColumn("max_to_median_ratio", F.round(F.col("max_rows") / F.col("median_rows"), 1))
)

# --- Step 3: pull the two scalars we need for tier classification --------------------
# .first() materializes a single Row to the driver. This is one of the few places we
# deliberately collect to the driver — we need the values to drive Python control flow.
summary_row = skew_summary.first()
max_to_median = float(summary_row["max_to_median_ratio"])
hot_share_pct = float(summary_row["hot_key_share_pct"])

print(f"Total fact rows: {total_rows:,}")
print(f"Hot-key share: {hot_share_pct}%")
print(f"Max/median ratio: {max_to_median}x")
print()
print("Skew summary:")
display(skew_summary)
print("Top 5 stores by row count:")
display(store_counts.orderBy(F.desc("row_count")).limit(5))

# --- Step 4: classify and recommend a tier -------------------------------------------
# Mirrors the decision table in the markdown above. We assume the workload includes a
# non-broadcast join on the skewed key (the worst common case); for pure aggregation
# workloads, drop one tier.
print()
print("=" * 60)
if max_to_median < 5:
    recommended_tier = "NONE"
    print("RECOMMENDATION: No skew handling needed.")
    print("Partial aggregation will absorb the imbalance. Move on.")
elif hot_share_pct > 80:
    recommended_tier = "SPLIT"
    print("RECOMMENDATION: Tier 3 — Split hot-key processing.")
    print(f"hot_share={hot_share_pct}% > 80%: even with salting, cold partitions")
    print("would be starved. Process the hot key as a dedicated query path.")
elif max_to_median > 50:
    recommended_tier = "SALT"
    print("RECOMMENDATION: Tier 2 — Salting.")
    print(f"max/median={max_to_median}x > 50x: AQE's default skewedPartitionFactor=5x")
    print("won't split aggressively enough. Use salting with N≈hot_share×num_partitions/1%.")
else:
    recommended_tier = "AQE"
    print("RECOMMENDATION: Tier 1 — Enable AQE skew-join.")
    print(f"max/median={max_to_median}x is in the 5–50x range AQE handles well.")
    print("Set spark.sql.adaptive.enabled=true and spark.sql.adaptive.skewJoin.enabled=true.")
print("=" * 60)


### Tier 1 — AQE skew-join (cheapest fix)

**When to use:** non-broadcast join on a key with `5x ≤ max/median ≤ 50x`. Pure aggregations don't benefit — AQE's skew handling is implemented for joins only.

**What AQE does:** at runtime, after the partial-aggregation/shuffle stage finishes writing map outputs, AQE reads the per-partition byte sizes. If a partition is both:
- larger than `spark.sql.adaptive.skewJoin.skewedPartitionFactor × median` (default **5x**), AND
- larger than `spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes` (default **256MB**),

it splits that partition into multiple sub-partitions on the read side and replicates the matching partition on the other side. The straggler reducer becomes N parallel reducers — same arithmetic as salting, but Catalyst handles it without code changes.

**Cost:** ~zero. Two config flags. AQE is on by default in Spark 3.2+ but `skewJoin.enabled` must be explicitly verified.

**Limits — the reasons we still need Tier 2/3:**
1. **Joins only.** A skewed `groupBy(key).countDistinct()` won't be helped — AQE can't split aggregation tasks the same way.
2. **The 256MB floor.** A skewed partition that's "only" 100MB but 100x larger than its peers won't trip AQE — the absolute threshold gates the split. You can lower it but don't go below ~64MB; sub-partition overhead starts to dominate.
3. **The 5x factor is gentle.** If `max/median` is 100x, AQE will split into ~20 sub-partitions, which still leaves each sub-task 5x heavier than median peers. Salting with explicit N can be more aggressive.

**Verifying AQE engaged:** run the query, then look at the SparkUI's SQL tab. You'll see `AQEShuffleRead` nodes in the executed plan with `isSkewedJoin = true` annotations on split partitions. `explain()` alone won't show this — AQE rewrites the plan at runtime, after each stage completes.

In [ ]:
# Tier 1 — AQE skew-join.
# Run this when the classifier above prints RECOMMENDATION: AQE.
# The "fix" is two config knobs; the rest of the query is unchanged from the un-skewed playbook.

# --- Configuration --------------------------------------------------------------------
# Print current values BEFORE we touch anything, so the diagnostic is honest about whether
# AQE was already on. (On Databricks runtime 9.1+ and OSS Spark 3.2+, both default true.)
print("Current AQE settings:")
for k in [
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.adaptive.skewJoin.skewedPartitionFactor",
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes",
    "spark.sql.adaptive.coalescePartitions.enabled",
]:
    print(f"  {k} = {spark.conf.get(k, '<unset>')}")

# Force-enable. Idempotent if already on.
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
# Lower the absolute-bytes threshold to 64MB. Default 256MB means a 100M-row × ~50 bytes
# fact partition for store_id=1 (~5GB → spread across many tasks) easily clears it; but
# for smaller datasets the 256MB floor blocks AQE entirely. 64MB is a safe lower bound.
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", str(64 * 1024 * 1024))

# --- Demo workload: a non-broadcast self-join on store_id ----------------------------
# We fabricate a skewed join scenario by self-joining the fact on store_id. Both sides are
# 100M rows so neither can be broadcast; the SortMergeJoin shuffles by store_id, which is
# exactly the skew vector AQE is built to handle.
#
# In a real codebase the equivalent would be a fact-to-fact join (returns vs sales),
# or a fact-to-large-dim join where the dim has too many rows to broadcast.
left = fact_sales.select("store_id", F.col("transaction_id").alias("txn_l"), "unit_price")
right = fact_sales.select("store_id", F.col("transaction_id").alias("txn_r"), "quantity")

aqe_join_demo = (
    left.join(right, "store_id", "inner")
    .groupBy("store_id")
    .agg(F.count("*").alias("pair_count"))
    .orderBy(F.desc("pair_count"))
    .limit(5)
)

# Static plan — AQE rewrites happen at runtime, so this won't show the skew split. It will
# show a SortMergeJoin (no broadcast) which is the necessary precondition for AQE skew handling.
print("\nStatic plan (pre-AQE-rewrite):")
aqe_join_demo.explain("formatted")

# Triggering execution. After this completes, open the SparkUI SQL tab → click the latest
# query → look for "AQEShuffleRead" nodes in the executed plan. Skewed partitions will be
# annotated with isSkewedJoin=true and split into multiple read sub-partitions.
print("\nExecuting (check SparkUI for AQEShuffleRead with isSkewedJoin=true):")
display(aqe_join_demo)


### Tier 2 — Salting (when AQE isn't enough)

**When to use:** AQE can't help (wide aggregations, windows) OR `max/median > 50x` and you need more aggressive splitting than AQE's default factor provides.

**The technique in three lines:**
1. Append a random salt `0..N-1` to the skewed key, turning `store_id=1` into N synthetic sub-keys.
2. Aggregate by the **composite** `(store_id, salt)` key. The hash partitioner spreads the N sub-keys across N reducers.
3. Drop the salt and aggregate again by `store_id` alone — over a tiny `200 × N` intermediate.

**Choosing N:** rule of thumb is `N ≈ (hot_key_share × num_partitions) / target_share_per_task`. With 50% hot share on 200 partitions and a 1% target, `N ≈ 100`. We use `N=16` below as a balanced demo value; if your classifier reports `hot_share_pct > 50%`, scale N up proportionally.

**Cost vs AQE:** salting forces an extra aggregation stage and requires code changes. AQE is free and automatic. So salting is a deliberate escalation, not a default.

In [ ]:
# Tier 2 — Salted revenue-by-store aggregation.
# Run this when the classifier prints RECOMMENDATION: SALT.

NUM_SALTS = 16  # see markdown above for sizing rationale.

fact_proj = fact_sales.select("store_id", "promotion_id", "quantity", "unit_price")
promo_proj = dim_promotion.select("promotion_id", "discount_pct")

enriched = (
    fact_proj
    .join(broadcast(promo_proj), "promotion_id", "left")
    .withColumn("net_revenue", net_revenue_expr())
    # Salt: pmod(rand()*N) gives a uniform integer in [0, N). Seeded for determinism.
    .withColumn("salt", F.pmod((F.rand(seed=99) * F.lit(NUM_SALTS)).cast("int"), F.lit(NUM_SALTS)))
)

# Step 1 — heavy aggregation on the COMPOSITE key. The hot store_id=1 gets spread across
# NUM_SALTS sub-keys, each landing on a different reducer.
salted_partial = (
    enriched
    .groupBy("store_id", "salt")
    .agg(F.sum("net_revenue").alias("salted_revenue"))
)

# Step 2 — collapse the salt dimension over the 200×16 = 3,200-row intermediate. Trivial.
revenue_by_store_salted = (
    salted_partial
    .groupBy("store_id")
    .agg(F.round(F.sum("salted_revenue"), 2).alias("total_net_revenue"))
    .orderBy(F.desc("total_net_revenue"))
)

# Plan checklist:
#   - Two HashAggregate pairs (partial+final) — one per groupBy.
#   - The intermediate Exchange between them carries ~3,200 rows across NUM_SALTS distinct
#     hash slots — load-balanced regardless of the original skew.
revenue_by_store_salted.explain("formatted")
display(revenue_by_store_salted.limit(10))


### Tier 3 — Split hot-key processing (last resort)

**When to use:** `hot_key_share > 80%`. At that point the dataset is essentially "the hot key + a long tail", and treating it as a single distribution wastes work.

**Why salting is no longer enough:** with `hot_key_share = 80%` and `NUM_SALTS = 16`, each salted sub-partition still carries `80% / 16 = 5%` of total rows — heavier than the median's `20% / 199 stores ≈ 0.1%`. Bumping `NUM_SALTS` to 200 fixes that ratio but creates a different problem: the cold partitions are now sliced into 200 sub-tasks each, so the cold side is task-scheduling-bound (lots of empty work) while the hot side still dominates wall-clock.

**The technique:**
1. **Filter the dataframe into two query paths**: `hot = df.filter(key == hot_value)` and `cold = df.filter(key != hot_value)`.
2. **Process each path with parallelism appropriate to its size.** Repartition the hot side to many tasks (often `200+`) so its single key gets a wide fan-out. Leave the cold side at its natural partitioning.
3. **`unionByName` the two results** at the end.

**Why this beats salting at extreme skew:**
- The hot path's repartition is essentially "salting just for the hot key" — but with N tuned to the hot key's actual size, not to a global compromise.
- The cold path runs without any salting overhead — its 199 cold keys are already well-distributed.
- Each path's plan is a textbook well-distributed query; no straggler on either side.

**Costs:** twice as much code, two query plans to reason about, and you must know the hot key's value at plan time (read it from the detection cell's output). Don't reach for this unless `hot_key_share > 80%` *and* salting hasn't fixed the wall-clock.

In [ ]:
# Tier 3 — Split hot-key processing.
# Run this when the classifier prints RECOMMENDATION: SPLIT.

# Hot-key value identified by the detection cell. Hardcoding it is acceptable here because
# we already collected the top-key result to the driver above; in a parameterized job, take
# this from a config or from the detection step's output.
HOT_STORE_ID = 1

# Hot-side parallelism — how many tasks should crunch through the hot key. Rule of thumb:
# match it to the cluster's available cores, or to ~(hot_rows / target_rows_per_task).
# At 50M hot rows and 500K rows/task, 100 is in the right neighborhood.
HOT_PARALLELISM = 100

# --- Common projections ---------------------------------------------------------------
fact_proj = fact_sales.select("store_id", "promotion_id", "quantity", "unit_price")
promo_proj = dim_promotion.select("promotion_id", "discount_pct")

# --- Compute net_revenue once, then split --------------------------------------------
# Doing the broadcast join + net_revenue derivation BEFORE the split avoids duplicating
# that work on each path. The split then only branches the aggregation, not the enrichment.
enriched = (
    fact_proj
    .join(broadcast(promo_proj), "promotion_id", "left")
    .withColumn("net_revenue", net_revenue_expr())
    .select("store_id", "net_revenue")
)

# --- HOT path: store_id = 1 -----------------------------------------------------------
# Filter first (pushed down to the file scan if the source is partitioned/Z-ordered on
# store_id; otherwise it's a cheap filter post-broadcast-join).
# repartition(HOT_PARALLELISM) shuffles ONLY the hot key's rows across HOT_PARALLELISM tasks.
# At ~50M hot rows, that's ~500K rows/task — same target as our normal partition size.
# Without this repartition, all 50M hot rows would funnel through whatever native partitioning
# the source has on store_id (often: one file → one task → one straggler).
hot_revenue = (
    enriched
    .filter(F.col("store_id") == F.lit(HOT_STORE_ID))
    .repartition(HOT_PARALLELISM)
    # Aggregate to a single row. Partial HashAggregate per task → 1 row each → final
    # aggregation reads HOT_PARALLELISM rows. Trivial final stage.
    .groupBy("store_id")
    .agg(F.sum("net_revenue").alias("total_net_revenue"))
)

# --- COLD path: every other store -----------------------------------------------------
# No repartition needed — the cold keys are already well-distributed (199 keys × ~250K rows
# each ≈ uniform). Standard groupBy.sum runs at full speed on this side.
cold_revenue = (
    enriched
    .filter(F.col("store_id") != F.lit(HOT_STORE_ID))
    .groupBy("store_id")
    .agg(F.sum("net_revenue").alias("total_net_revenue"))
)

# --- Stitch back together -------------------------------------------------------------
# unionByName instead of union: defensive against accidental column-order drift between the
# two paths. Round at the end so both paths share the same numeric formatting.
revenue_by_store_split = (
    hot_revenue
    .unionByName(cold_revenue)
    .withColumn("total_net_revenue", F.round("total_net_revenue", 2))
    .orderBy(F.desc("total_net_revenue"))
)

# Plan checklist:
#   - Two distinct branches under the union — each branch is its own well-balanced query.
#   - The hot branch shows a Repartition(100) feeding the HashAggregate.
#   - The cold branch has NO repartition — natural distribution is fine.
#   - The Union at the top has no shuffle — it's a logical concatenation.
revenue_by_store_split.explain("formatted")
display(revenue_by_store_split.limit(10))


## Non-broadcastable dim — `dim_customer` at 200M rows

`dim_customer` has grown from 100K rows to 200M. At ~30 bytes/row that's ~6GB serialized — about 600x the default broadcast threshold (`spark.sql.autoBroadcastJoinThreshold = 10MB`). Forcing `broadcast(dim_customer)` would:

1. **OOM the driver** during the broadcast `collect` (the driver pulls the entire dim into a single JVM heap).
2. **OOM every executor** that successfully receives it (the broadcast hash table has to fit in each executor's heap, alongside everything else the task is doing).

So the broadcast playbook from the earlier metrics no longer applies for `customer_id` joins. Three things change:

### 1. Join strategy: BroadcastHashJoin → SortMergeJoin
Without `broadcast()`, Catalyst falls back to **SortMergeJoin (SMJ)**: shuffle BOTH the fact and the dim by `customer_id`, sort each side within partition, then stream the two sorted runs together. Cost: ~5–10GB shuffle on the fact + ~6GB on the dim, every query. That's the price of admission once the dim is too large to broadcast.

### 2. Bucketing eliminates the shuffle
`synthetic_sales.py` now writes both `fact_sales` and `dim_customer` with `bucketBy(32, "customer_id").sortBy("customer_id")`. Same bucket count + same key + same hash function on both sides means Catalyst recognizes at plan time that matching `customer_id` values are already co-located in matching files. The plan reduces from `Exchange → Sort → SMJ` (per side) to just `SMJ` over pre-bucketed inputs — **the shuffle and sort are skipped entirely**. This is the closest you get to broadcast performance for a non-broadcastable dim.

### 3. Project hard, push filters
The other levers from the earlier playbook still help, just more so:
- **Project before join**: the SMJ now ships post-projection rows through the merge — fewer columns = smaller spill, smaller network if the bucketing assumption ever fails.
- **Push predicates below the join**: Spark's predicate pushdown will move `WHERE` clauses on either side into the file scan, before the SMJ. At 200M dim rows this is the difference between sorting 200M rows and sorting (e.g.) 50M.
- **Aggregate on the fact's side BEFORE the join when possible**: if the metric only needs `loyalty_tier` from the customer side, pre-aggregating the fact by `customer_id` reduces the SMJ input from 100M rows to 100K-ish (one per active customer), making the heaviest stage trivial.

The cell below demonstrates strategy #3 — pre-aggregate, then SMJ — for a "revenue by loyalty tier" metric.

In [ ]:
# Revenue by loyalty tier.
# Joins: fact -> dim_promotion (broadcast, tiny) AND fact -> dim_customer (SMJ, 200M rows).
# This is the demo of the non-broadcast path.
#
# Strategy: PRE-AGGREGATE the fact down to per-customer revenue BEFORE the SMJ. The SMJ then
# runs over (≤NUM_CUSTOMERS rows) on the fact side instead of 100M rows. This is the single
# most impactful optimization once a dim outgrows the broadcast threshold.

# Step 1: project fact to ONLY the columns needed for the per-customer aggregation. The SMJ
# input will inherit this projection.
fact_proj = fact_sales.select("customer_id", "promotion_id", "quantity", "unit_price")

# Step 2: dim_promotion stays broadcast — it's still tiny.
promo_proj = dim_promotion.select("promotion_id", "discount_pct")

# Step 3: project dim_customer to (join_key + only what we need). loyalty_tier is the ONLY
# customer column this metric uses. At 200M rows, dropping every other column cuts the SMJ's
# right-side payload by ~70% and meaningfully reduces sort/spill memory.
customer_proj = dim_customer.select("customer_id", "loyalty_tier")

# Step 4: enrich + pre-aggregate by customer_id.
# This is the heavy stage on the fact (100M rows in, ~100K rows out — assuming most of our
# 200M customers are inactive). The groupBy collapses 100M rows to ≤NUM_CUSTOMERS rows
# BEFORE the expensive customer SMJ, which is why this restructuring matters so much.
revenue_per_customer = (
    fact_proj
    .join(broadcast(promo_proj), "promotion_id", "left")  # tiny dim, still broadcastable
    .withColumn("net_revenue", net_revenue_expr())
    .groupBy("customer_id")
    .agg(F.sum("net_revenue").alias("customer_revenue"))
)

# Step 5: SortMergeJoin to dim_customer. NOTE: no broadcast() — dim_customer is 6GB.
# With both fact_sales AND dim_customer bucketed on customer_id (32 buckets each, sorted
# within bucket), Spark's plan SHOULD show no Exchange/Sort before the SMJ — the bucketing
# co-locates matching keys. Verify this in explain("formatted") output below.
#
# If you ever see Exchange nodes feeding both sides of the SMJ, something has invalidated
# the bucketing assumption (different bucket counts, different hash, AQE coalescing partitions,
# or the planner deciding to ignore bucketing). Common fix: SET spark.sql.sources.bucketing.enabled=true
# and SET spark.sql.adaptive.coalescePartitions.enabled=false for this query.
revenue_by_loyalty_tier = (
    revenue_per_customer
    .join(customer_proj, "customer_id", "inner")  # inner: drop fact rows for non-existent customers
    # Step 6: re-aggregate by loyalty_tier. Input is at most ~NUM_CUSTOMERS rows, so this is
    # essentially free — partial HashAggregate per task collapses to 4 rows (one per tier).
    .groupBy("loyalty_tier")
    .agg(F.round(F.sum("customer_revenue"), 2).alias("total_net_revenue"))
    .orderBy(F.desc("total_net_revenue"))
)

# Plan checklist:
#   - BroadcastHashJoin on dim_promotion (tiny).
#   - HashAggregate (partial → final) collapsing fact to per-customer revenue.
#   - SortMergeJoin between revenue_per_customer and dim_customer on customer_id.
#     IDEAL: no Exchange/Sort feeding the SMJ — bucketed inputs are pre-shuffled and pre-sorted.
#     ACCEPTABLE: Exchange on the LEFT side only (revenue_per_customer is the output of a
#     groupBy, which already shuffled by customer_id, so it should match the bucket layout).
#     RED FLAG: Exchange on BOTH sides — bucketing isn't being used, every query pays full SMJ cost.
#   - HashAggregate by loyalty_tier — trivial.
revenue_by_loyalty_tier.explain("formatted")
display(revenue_by_loyalty_tier)
